In [1]:
import kagglehub
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping


c:\Users\Mega\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("pacificrm/skindiseasedataset")

print("Path to dataset files:", path)

Resuming download from 168820736 bytes (1292100380 bytes left)...
Resuming download to C:\Users\Mega\.cache\kagglehub\datasets\pacificrm\skindiseasedataset\6.archive (168820736/1460921116) bytes left.


100%|██████████| 1.36G/1.36G [18:58<00:00, 1.14MB/s] 

Extracting files...


Path to dataset files: C:\Users\Mega\.cache\kagglehub\datasets\pacificrm\skindiseasedataset\versions\6


In [3]:
test_path='./SkinDisease/test'
test_ds= tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

print("Test classes:", test_ds.class_names)

Found 1546 files belonging to 22 classes.
Test classes: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 'Vascular_Tumors', 'Vasculitis', 'Vitiligo', 'Warts']


In [14]:
train_path='./SkinDisease/train'
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

print("Classes:", train_ds.class_names)

Found 13898 files belonging to 22 classes.
Classes: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 'Vascular_Tumors', 'Vasculitis', 'Vitiligo', 'Warts']


In [15]:
normalization = tf.keras.layers.Normalization()
normalization.adapt(
    train_ds.map(lambda x, y: x)
)

In [16]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [30]:
model = tf.keras.Sequential([
        layers.Input(shape=(224, 224, 3)),
        
    # Standardization
    normalization,

    # Data Augmentation
    data_augmentation,

    # CNN
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(len(train_ds.class_names), activation="softmax")
])


model.compile(
    optimizer=tf.keras.optimizers.Nadam(
        learning_rate=0.0001,
        weight_decay=0.0001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-07,
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)



training_history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=[early_stop]
)




Epoch 1/10
421/869 ━━━━━━━━━━━━━━━━━━━━ 1:59 267ms/step - accuracy: 0.1467 - loss: 2.8584

KeyboardInterrupt: 

In [26]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 127ms/step - accuracy: 0.3829 - loss: 2.0999
Test Loss: 2.0999369621276855
Test Accuracy: 0.3829236626625061
